[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/01-python/06_lists.ipynb)

*Python from Scratch · Notebook 6 of 12*

# Lists

Ordered, changeable collections: indexing, methods, slicing, copying, sorting and lists of lists.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A list is a row of numbered lockers. Reaching any locker is instant, but squeezing one into the middle means shifting everyone along.

**What interviewers really test here:** Costs of append vs insert, in-place vs copy, slicing, and aliasing bugs.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Creating a list and reading items *(Basics)*
- [ ] 2. Changing a list *(Basics)*
- [ ] 3. Two names, one list (aliasing) *(Core)*
- [ ] 4. Looping over lists and building new ones *(Core)*
- [ ] 5. Sorting *(Core)*
- [ ] 6. Lists of lists (2-D lists) *(Advanced)*
- [ ] 7. Performance of list operations *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

A **list** is an **ordered**, **mutable** collection that can hold items of any type, including duplicates. It is written with square brackets: `[1, "two", 3.0]`.




- **Ordered** — items keep the position you gave them and are accessed by index.
- **Mutable** — items can be added, removed or replaced after creation.
- **Heterogeneous** — different types can be mixed.
- **Dynamic** — grows and shrinks automatically.



Internally CPython implements a list as a **dynamic array of references** to objects.

### Creating lists

| Way | Example |
| --- | --- |
| Literal | `[1, 2, 3]`, `[]` |
| From an iterable | `list("abc")`, `list(range(5))` |
| Repetition | `[0] * 5` |
| Comprehension | `[x * x for x in range(5)]` |
| From splitting | `"a,b".split(",")` |

### Methods and operations

| Method / operation | Description | Time |
| --- | --- | --- |
| `append(x)` | add to the end | O(1)\* |
| `extend(it)` / `+=` | add all items of an iterable | O(k) |
| `insert(i, x)` | insert before index i | O(n) |
| `remove(x)` | remove first x, ValueError if absent | O(n) |
| `pop()` / `pop(i)` | remove and return last / i-th item | O(1) / O(n) |
| `clear()` | remove all items | O(n) |
| `index(x)` | first position of x | O(n) |
| `count(x)` | number of x | O(n) |
| `sort(key=, reverse=)` | sort in place (stable Timsort) | O(n log n) |
| `reverse()` | reverse in place | O(n) |
| `copy()` | shallow copy | O(n) |
| `a[i]`, `len(a)` | index, length | O(1) |
| `x in a` | membership | O(n) |
| `a[i:j:k]` | slice (new list) | O(slice length) |
| `del a[i:j]` / slice assignment | delete or replace a range | O(n) |



\* amortised. Useful built-ins: `len, sum, min, max, sorted, reversed, enumerate, zip, any, all`.

### References, aliasing and copying

A variable holds a **reference** to a list object. Assigning or passing a list copies the reference, not the list.



| Operation | New outer list? | Inner objects |
| --- | --- | --- |
| `b = a` | no (alias) | shared |
| `a[:]`, `a.copy()`, `list(a)` | yes (**shallow copy**) | shared |
| `copy.deepcopy(a)` | yes (**deep copy**) | copied recursively |



Pitfalls caused by aliasing:




- `[[0] * 3] * 2` creates two references to one inner list.
- A mutable **default argument** (`def f(x, acc=[])`) is shared between calls.
- Functions that modify a list argument change the caller's list.

### List vs other sequences

|   | list | tuple | array.array | collections.deque |
| --- | --- | --- | --- | --- |
| Mutable | yes | no | yes | yes |
| Item types | any | any | one numeric type | any |
| Fast ends | right end | — | right end | **both ends** |
| Hashable | no | yes (if items are) | no | no |
| Typical use | general collection | fixed records, dict keys | compact numbers | queues |



For heavy numeric work, NumPy arrays are much faster and smaller than lists.

### Applications

- Storing ordered data: records read from a file, user inputs, search results.
- Stacks (`append`/`pop`) for undo or backtracking.
- Matrices and grids as lists of lists (games, spreadsheets).
- Building results step by step before joining or saving them.

---
# Part 2: Lessons, step by step

## Lesson 1: Creating a list and reading items

*Level: Basics*

A **list** stores many values in one variable, in order:



```python
marks = [72, 85, 90]
mixed = ["Ana", 21, True, 3.5]    # items can be any type
empty = []
```



Items are numbered from 0, just like string characters: `marks[0]` is 72, `marks[-1]` is the last item. `len(marks)` is the number of items and `85 in marks` checks membership.



Slicing works too: `marks[0:2]` → `[72, 85]`. A slice is a **new** list.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def list_basics_demo():
    marks = [72, 85, 90]
    first = marks[0]
    last = marks[-1]
    size = len(marks)
    top_two = marks[1:]
    has_85 = 85 in marks
    print(first, last, size, top_two, has_85)


list_basics_demo()

72 90 3 [85, 90] True


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2). Try one now, or carry on and do them in Part 3.

## Lesson 2: Changing a list

*Level: Basics*

Unlike strings, lists are **mutable** — you can change them in place:



| Code | Effect |
| --- | --- |
| `a[1] = 50` | replace item 1 |
| `a.append(x)` | add x at the end |
| `a.insert(i, x)` | insert x before position i |
| `a.extend([x, y])` | add several items |
| `a.remove(x)` | delete the first x (ValueError if missing) |
| `a.pop()` / `a.pop(i)` | delete and **return** the last / i-th item |
| `del a[i]` | delete item i |
| `a.clear()` | remove everything |

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def modify_demo():
    cart = ["milk", "eggs"]
    cart.append("bread")
    cart.insert(0, "tea")
    cart[2] = "butter"
    removed = cart.pop()
    cart.remove("tea")
    cart.extend(["rice", "salt"])
    print(cart, "removed:", removed)


modify_demo()

['milk', 'butter', 'rice', 'salt'] removed: bread


**✏️ Your turn:** you now have what you need for [Q10](#q10), [Q19](#q19). Try one now, or carry on and do them in Part 3.

## Lesson 3: Two names, one list (aliasing)

*Level: Core*

This surprises every beginner. Assignment does **not** copy a list — it adds another name for the same list:



```python
a = [1, 2, 3]
b = a          # b and a are the SAME list
b.append(4)
print(a)       # [1, 2, 3, 4]  ← a changed too!
```



To get an independent copy use `a[:]`, `a.copy()` or `list(a)`. For lists that contain other lists, those make a **shallow** copy (the inner lists are still shared) — use `copy.deepcopy(a)` for a full copy.



The same thing happens when you pass a list to a function: the function can change the caller's list.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def aliasing_demo():
    a = [1, 2, 3]
    b = a
    b.append(4)
    c = a[:]
    c.append(99)
    print("a =", a, "| b =", b, "| c =", c)
    print("a is b:", a is b, "| a is c:", a is c)


aliasing_demo()

a = [1, 2, 3, 4] | b = [1, 2, 3, 4] | c = [1, 2, 3, 4, 99]
a is b: True | a is c: False


**✏️ Your turn:** you now have what you need for [Q4](#q4). Try one now, or carry on and do them in Part 3.

## Lesson 4: Looping over lists and building new ones

*Level: Core*

Loop directly over the items, or use `enumerate` when you also need the position:



```python
for price in prices:
    ...
for i, price in enumerate(prices):
    ...
```



A very common task is building a new list from an old one. The long way uses a loop and `append`; the short way is a **list comprehension**:



```python
doubled = []
for x in nums:
    doubled.append(x * 2)

doubled = [x * 2 for x in nums]              # same result
evens = [x for x in nums if x % 2 == 0]     # with a filter
```

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [5]:
def building_demo():
    nums = [3, 8, 5, 12]
    doubled = []
    for x in nums:
        doubled.append(x * 2)
    evens = [x for x in nums if x % 2 == 0]
    for i, x in enumerate(nums):
        if x > 6:
            print("index", i, "has", x)
    print(doubled, evens)


building_demo()

index 1 has 8
index 3 has 12
[6, 16, 10, 24] [8, 12]


**✏️ Your turn:** you now have what you need for [Q13](#q13), [Q8](#q8), [Q3](#q3), [Q18](#q18). Try one now, or carry on and do them in Part 3.

## Lesson 5: Sorting

*Level: Core*

|   | `list.sort()` | `sorted(iterable)` |
| --- | --- | --- |
| Changes the original? | yes, in place | no, returns a new list |
| Returns | `None` | the sorted list |
| Works on | lists only | any iterable (strings, tuples, dicts…) |



Both accept `reverse=True` and a `key` function that says what to sort by:



```python
words.sort(key=len)                          # by length
people.sort(key=lambda p: p[1])              # by the second field
sorted(names, key=str.lower)                 # case-insensitive
```



**⚠️ Watch out:** `nums = nums.sort()` sets nums to None. Use either `nums.sort()` or `nums = sorted(nums)`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [6]:
def sorting_demo():
    nums = [5, 2, 9, 1]
    new = sorted(nums)
    nums.sort(reverse=True)
    words = ["kiwi", "fig", "banana"]
    by_len = sorted(words, key=len)
    result = nums.sort()
    print(new, nums, by_len, result)


sorting_demo()

[1, 2, 5, 9] [1, 2, 5, 9] ['fig', 'kiwi', 'banana'] None


**✏️ Your turn:** you now have what you need for [Q15](#q15), [Q12](#q12), [Q17](#q17). Try one now, or carry on and do them in Part 3.

## Lesson 6: Lists of lists (2-D lists)

*Level: Advanced*

A list can hold other lists — a simple way to represent a table or grid. `grid[r][c]` is row r, column c.



```python
grid = [[1, 2, 3],
        [4, 5, 6]]
grid[1][2]          # 6
for row in grid:
    for x in row:
        ...
```



**⚠️ Watch out:** Create grids with a comprehension: `[[0] * cols for _ in range(rows)]`. Writing `[[0] * cols] * rows` repeats the **same** row object, so changing one row changes all of them.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [7]:
def grid_demo():
    wrong = [[0] * 3] * 2
    wrong[0][0] = 9
    right = [[0] * 3 for _ in range(2)]
    right[0][0] = 9
    print("wrong:", wrong)
    print("right:", right)


grid_demo()

wrong: [[9, 0, 0], [9, 0, 0]]
right: [[9, 0, 0], [0, 0, 0]]


**✏️ Your turn:** you now have what you need for [Q5](#q5), [Q20](#q20), [Q7](#q7). Try one now, or carry on and do them in Part 3.

## Lesson 7: Performance of list operations

*Level: Advanced*

A Python list is a **dynamic array**: items sit side by side in memory with some spare room at the end.



| Operation | Cost |
| --- | --- |
| `a[i]`, `a[i] = x`, `len(a)` | O(1) |
| `append`, `pop()` | O(1) (amortised) |
| `insert(0, x)`, `pop(0)`, `remove(x)` | O(n) — items shift |
| `x in a`, `a.index(x)`, `a.count(x)` | O(n) — scans |
| `a[i:j]`, `a + b`, `a.copy()` | O(length of result) |
| `sort`, `sorted` | O(n log n) |



That's why the "brute force" answers that call `in`, `remove` or `insert(0, …)` inside a loop are O(n²), while the optimal ones use a set, a dict or a single pass.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [8]:
def stack_demo():
    stack = []
    for page in ["home", "search", "result"]:
        stack.append(page)
    back = stack.pop()
    print("went back from", back, "to", stack[-1])


stack_demo()

went back from result to search


**✏️ Your turn:** you now have what you need for [Q16](#q16), [Q11](#q11). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Sum of a List

🟢 Warm-up · *Accumulator vs sum()*

Return the sum of the numbers in a list.

**Examples**

- `nums = [4, 8, 15]` → `27`
- `nums = []` → `0`
- `nums = [-2, 2]` → `0`

In [ ]:
def list_sum(nums):
    """
    Return the sum of the numbers in a list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(list_sum, [
    (([4, 8, 15],), 27),
    (([],), 0),
    (([-2, 2],), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over indices and add.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The built-in `sum()`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop over indices and add.

```python
def list_sum(nums):
    total = 0
    for i in range(len(nums)):
        total = total + nums[i]
    return total
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** The built-in `sum()`.

```python
def list_sum(nums):
    return sum(nums)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over indices and add. That costs O(n) time and O(1) space."

"I can do better: The built-in `sum()`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/sum-of-a-list)

<a id="q2"></a>
### Q2. Average of a List

🟢 Warm-up · *sum / len*

Return the average of a list rounded to 2 decimals, or 0 for an empty list.

**Examples**

- `nums = [2, 4, 9]` → `5.0`
- `nums = [1, 2]` → `1.5`
- `nums = []` → `0`

In [ ]:
def average(nums):
    """
    Return the average of a list rounded to 2 decimals, or 0 for an empty list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(average, [
    (([2, 4, 9],), 5.0),
    (([1, 2],), 1.5),
    (([],), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count and add in one loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`sum(nums) / len(nums)` with an empty-list guard.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Count and add in one loop.

```python
def average(nums):
    total, count = 0, 0
    for x in nums:
        total += x
        count += 1
    if count == 0:
        return 0
    return round(total / count, 2)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `sum(nums) / len(nums)` with an empty-list guard.

```python
def average(nums):
    return round(sum(nums) / len(nums), 2) if nums else 0
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count and add in one loop. That costs O(n) time and O(1) space."

"I can do better: `sum(nums) / len(nums)` with an empty-list guard. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/average-of-a-list)

<a id="q3"></a>
### Q3. Remove Duplicates, Keep Order

🟢 Warm-up · *in vs dict*

Return a new list with duplicates removed, keeping the first occurrence of each value in order.

**Examples**

- `items = [3, 1, 3, 2, 1]` → `[3, 1, 2]`
- `items = []` → `[]`
- `items = ['a', 'a']` → `['a']`

In [ ]:
def dedupe(items):
    """
    Return a new list with duplicates removed, keeping the first occurrence of
    each value in order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(dedupe, [
    (([3, 1, 3, 2, 1],), [3, 1, 2]),
    (([],), []),
    ((['a', 'a'],), ['a']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Append each value if it's not already in the result list (each `in` scans the list).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`dict.fromkeys` keeps the first occurrence and insertion order; checking a dict is O(1).

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Append each value if it's not already in the result list (each `in` scans the list).

```python
def dedupe(items):
    result = []
    for x in items:
        if x not in result:
            result.append(x)
    return result
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `dict.fromkeys` keeps the first occurrence and insertion order; checking a dict is O(1).

```python
def dedupe(items):
    return list(dict.fromkeys(items))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Append each value if it's not already in the result list (each `in` scans the list). That costs O(n²) time and O(n) space."

"I can do better: `dict.fromkeys` keeps the first occurrence and insertion order; checking a dict is O(1). That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/remove-duplicates-keep-order)

<a id="q4"></a>
### Q4. Reverse a List in Place

🟢 Warm-up · *Swapping*

Reverse the list **in place** (change the given list) and return it.

**Examples**

- `items = [1, 2, 3, 4]` → `[4, 3, 2, 1]`
- `items = [7]` → `[7]`
- `items = []` → `[]`

In [ ]:
def reverse_in_place(items):
    """
    Reverse the list in place (change the given list) and return it.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(reverse_in_place, [
    (([1, 2, 3, 4],), [4, 3, 2, 1]),
    (([7],), [7]),
    (([],), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Swap the first and last, then move inward, with explicit indices.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`items.reverse()` does exactly this.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Swap the first and last, then move inward, with explicit indices.

```python
def reverse_in_place(items):
    i, j = 0, len(items) - 1
    while i < j:
        temp = items[i]
        items[i] = items[j]
        items[j] = temp
        i += 1
        j -= 1
    return items
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `items.reverse()` does exactly this.

```python
def reverse_in_place(items):
    items.reverse()
    return items
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Swap the first and last, then move inward, with explicit indices. That costs O(n) time and O(1) space."

"I can do better: `items.reverse()` does exactly this. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/reverse-a-list-in-place)

<a id="q5"></a>
### Q5. Flatten a List of Lists

🟢 Warm-up · *Nested loops vs comprehension*

Turn `[[1, 2], [3], [4, 5]]` into `[1, 2, 3, 4, 5]`.

**Examples**

- `grid = [[1, 2], [3], [4, 5]]` → `[1, 2, 3, 4, 5]`
- `grid = [[], [1]]` → `[1]`

In [ ]:
def flatten(grid):
    """
    Turn [[1, 2], [3], [4, 5]] into [1, 2, 3, 4, 5].
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(flatten, [
    (([[1, 2], [3], [4, 5]],), [1, 2, 3, 4, 5]),
    (([[], [1]],), [1]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two nested loops appending each item.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A nested list comprehension reads in the same order as the loops.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Two nested loops appending each item.

```python
def flatten(grid):
    flat = []
    for row in grid:
        for x in row:
            flat.append(x)
    return flat
```

**Cost:** O(total) time, O(total) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A nested list comprehension reads in the same order as the loops.

```python
def flatten(grid):
    return [x for row in grid for x in row]
```

**Cost:** O(total) time, O(total) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two nested loops appending each item. That costs O(total) time and O(total) space."

"I can do better: A nested list comprehension reads in the same order as the loops. That's O(total) time and O(total) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/flatten-a-list-of-lists)

<a id="q6"></a>
### Q6. Count Occurrences

🟢 Warm-up · *count()*

Return how many times `value` appears in the list.

**Examples**

- `items = [1, 2, 1, 1], value = 1` → `3`
- `items = ['a', 'b'], value = 'c'` → `0`

In [ ]:
def occurrences(items, value):
    """
    Return how many times value appears in the list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(occurrences, [
    (([1, 2, 1, 1], 1), 3),
    ((['a', 'b'], 'c'), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop and count matches.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`items.count(value)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop and count matches.

```python
def occurrences(items, value):
    count = 0
    for x in items:
        if x == value:
            count += 1
    return count
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `items.count(value)`.

```python
def occurrences(items, value):
    return items.count(value)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop and count matches. That costs O(n) time and O(1) space."

"I can do better: `items.count(value)`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/count-occurrences)

<a id="q7"></a>
### Q7. Split Into Chunks

🟢 Warm-up · *Slicing with a step*

Split a list into pieces of size k (the last piece may be shorter).

**Examples**

- `items = [1, 2, 3, 4, 5], k = 2` → `[[1, 2], [3, 4], [5]]`
- `items = [], k = 3` → `[]`
- `items = [1, 2], k = 5` → `[[1, 2]]`

In [ ]:
def chunks(items, k):
    """
    Split a list into pieces of size k (the last piece may be shorter).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(chunks, [
    (([1, 2, 3, 4, 5], 2), [[1, 2], [3, 4], [5]]),
    (([], 3), []),
    (([1, 2], 5), [[1, 2]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Collect items into a current chunk; when it's full, store it and start a new one.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Slices `items[i:i + k]` for i in `range(0, n, k)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Collect items into a current chunk; when it's full, store it and start a new one.

```python
def chunks(items, k):
    result, current = [], []
    for x in items:
        current.append(x)
        if len(current) == k:
            result.append(current)
            current = []
    if current:
        result.append(current)
    return result
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Slices `items[i:i + k]` for i in `range(0, n, k)`.

```python
def chunks(items, k):
    return [items[i:i + k] for i in range(0, len(items), k)]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Collect items into a current chunk; when it's full, store it and start a new one. That costs O(n) time and O(n) space."

"I can do better: Slices `items[i:i + k]` for i in `range(0, n, k)`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/split-into-chunks)

<a id="q8"></a>
### Q8. Running Total

🟢 Warm-up · *Prefix sums*

Return the running totals: [1, 2, 3, 4] → [1, 3, 6, 10].

**Examples**

- `nums = [1, 2, 3, 4]` → `[1, 3, 6, 10]`
- `nums = [5]` → `[5]`
- `nums = []` → `[]`

In [ ]:
def running_total(nums):
    """
    Return the running totals: [1, 2, 3, 4] → [1, 3, 6, 10].
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(running_total, [
    (([1, 2, 3, 4],), [1, 3, 6, 10]),
    (([5],), [5]),
    (([],), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each position, sum the slice up to it from scratch.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep one running total and append it each step (or `itertools.accumulate`).

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** For each position, sum the slice up to it from scratch.

```python
def running_total(nums):
    result = []
    for i in range(len(nums)):
        result.append(sum(nums[:i + 1]))
    return result
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Keep one running total and append it each step (or `itertools.accumulate`).

```python
def running_total(nums):
    result, total = [], 0
    for x in nums:
        total += x
        result.append(total)
    return result
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each position, sum the slice up to it from scratch. That costs O(n²) time and O(n) space."

"I can do better: Keep one running total and append it each step (or `itertools.accumulate`). That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/running-total)

<a id="q9"></a>
### Q9. Product of a List

🟢 Warm-up · *math.prod*

Return the product of all numbers (1 for an empty list).

**Examples**

- `nums = [2, 3, 4]` → `24`
- `nums = []` → `1`
- `nums = [5, 0, 2]` → `0`

In [ ]:
def product(nums):
    """
    Return the product of all numbers (1 for an empty list).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(product, [
    (([2, 3, 4],), 24),
    (([],), 1),
    (([5, 0, 2],), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Multiply in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`math.prod(nums)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Multiply in a loop.

```python
def product(nums):
    result = 1
    for x in nums:
        result *= x
    return result
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `math.prod(nums)`.

```python
def product(nums):
    import math
    return math.prod(nums)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Multiply in a loop. That costs O(n) time and O(1) space."

"I can do better: `math.prod(nums)`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/product-of-a-list)

<a id="q10"></a>
### Q10. Remove Every Copy of a Value

🟢 Warm-up · *remove() vs comprehension*

Return a new list without any occurrence of `value`.

**Examples**

- `items = [1, 2, 1, 3, 1], value = 1` → `[2, 3]`
- `items = [2], value = 1` → `[2]`

In [ ]:
def without(items, value):
    """
    Return a new list without any occurrence of value.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(without, [
    (([1, 2, 1, 3, 1], 1), [2, 3]),
    (([2], 1), [2]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Copy the list and call `remove` while the value is still in it (each call scans).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep only the other values with a comprehension.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Copy the list and call `remove` while the value is still in it (each call scans).

```python
def without(items, value):
    result = items[:]
    while value in result:
        result.remove(value)
    return result
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Keep only the other values with a comprehension.

```python
def without(items, value):
    return [x for x in items if x != value]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Copy the list and call `remove` while the value is still in it (each call scans). That costs O(n²) time and O(n) space."

"I can do better: Keep only the other values with a comprehension. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/remove-every-copy-of-a-value)

<a id="q11"></a>
### Q11. Index of the Maximum

🟢 Warm-up · *max with key*

Return the index of the largest value (first one if repeated).

**Examples**

- `nums = [3, 9, 2, 9]` → `1`
- `nums = [5]` → `0`
- `nums = [-1, -3]` → `0`

In [ ]:
def argmax(nums):
    """
    Return the index of the largest value (first one if repeated).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(argmax, [
    (([3, 9, 2, 9],), 1),
    (([5],), 0),
    (([-1, -3],), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track the best index while looping.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`max(range(len(nums)), key=nums.__getitem__)`, or `nums.index(max(nums))`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Track the best index while looping.

```python
def argmax(nums):
    best = 0
    for i in range(1, len(nums)):
        if nums[i] > nums[best]:
            best = i
    return best
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `max(range(len(nums)), key=nums.__getitem__)`, or `nums.index(max(nums))`.

```python
def argmax(nums):
    return nums.index(max(nums))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track the best index while looping. That costs O(n) time and O(1) space."

"I can do better: `max(range(len(nums)), key=nums.__getitem__)`, or `nums.index(max(nums))`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/index-of-the-maximum)

<a id="q12"></a>
### Q12. Is the List Sorted?

🟢 Warm-up · *Neighbour pairs*

Return True if the list is in non-decreasing order.

**Examples**

- `nums = [1, 2, 2, 5]` → `True`
- `nums = [3, 1]` → `False`
- `nums = []` → `True`

In [ ]:
def is_sorted(nums):
    """
    Return True if the list is in non-decreasing order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_sorted, [
    (([1, 2, 2, 5],), True),
    (([3, 1],), False),
    (([],), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort a copy and compare.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Every neighbour pair must be in order; `all` stops at the first bad pair.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Sort a copy and compare.

```python
def is_sorted(nums):
    return nums == sorted(nums)
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Every neighbour pair must be in order; `all` stops at the first bad pair.

```python
def is_sorted(nums):
    return all(nums[i] <= nums[i + 1] for i in range(len(nums) - 1))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort a copy and compare. That costs O(n log n) time and O(n) space."

"I can do better: Every neighbour pair must be in order; `all` stops at the first bad pair. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/is-the-list-sorted)

<a id="q13"></a>
### Q13. Separate Evens and Odds

🟢 Warm-up · *Building two lists*

Return `(evens, odds)` keeping the original order in each.

**Examples**

- `nums = [1, 2, 3, 4, 5]` → `([2, 4], [1, 3, 5])`
- `nums = []` → `([], [])`

In [ ]:
def split_parity(nums):
    """
    Return (evens, odds) keeping the original order in each.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(split_parity, [
    (([1, 2, 3, 4, 5],), ([2, 4], [1, 3, 5])),
    (([],), ([], [])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two separate loops, one per list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One loop that appends to the right list.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Two separate loops, one per list.

```python
def split_parity(nums):
    evens = []
    for x in nums:
        if x % 2 == 0:
            evens.append(x)
    odds = []
    for x in nums:
        if x % 2 != 0:
            odds.append(x)
    return evens, odds
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** One loop that appends to the right list.

```python
def split_parity(nums):
    evens, odds = [], []
    for x in nums:
        (evens if x % 2 == 0 else odds).append(x)
    return evens, odds
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two separate loops, one per list. That costs O(n) time and O(n) space."

"I can do better: One loop that appends to the right list. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/separate-evens-and-odds)

<a id="q14"></a>
### Q14. Shopping Cart Total

🟢 Warm-up · *List of pairs*

`cart` is a list of (price, quantity) pairs. Return the total cost rounded to 2 decimals.

**Examples**

- `cart = [(2.5, 4), (10, 1)]` → `20.0`
- `cart = []` → `0`
- `cart = [(0.99, 3)]` → `2.97`

In [ ]:
def cart_total(cart):
    """
    cart is a list of (price, quantity) pairs. Return the total cost rounded to
    2 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cart_total, [
    (([(2.5, 4), (10, 1)],), 20.0),
    (([],), 0),
    (([(0.99, 3)],), 2.97),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop with indices into each pair.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Unpack each pair in a generator inside `sum`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop with indices into each pair.

```python
def cart_total(cart):
    total = 0
    for i in range(len(cart)):
        price = cart[i][0]
        qty = cart[i][1]
        total += price * qty
    return round(total, 2)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Unpack each pair in a generator inside `sum`.

```python
def cart_total(cart):
    return round(sum(price * qty for price, qty in cart), 2)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop with indices into each pair. That costs O(n) time and O(1) space."

"I can do better: Unpack each pair in a generator inside `sum`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/shopping-cart-total)

<a id="q15"></a>
### Q15. Second Largest

🟡 Interview standard · *One pass*

Return the second largest distinct value, or None if it doesn't exist.

**Examples**

- `nums = [4, 9, 2, 9, 7]` → `7`
- `nums = [5, 5]` → `None`
- `nums = [1, 2]` → `1`

In [ ]:
def second_largest(nums):
    """
    Return the second largest distinct value, or None if it doesn't exist.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(second_largest, [
    (([4, 9, 2, 9, 7],), 7),
    (([5, 5],), None),
    (([1, 2],), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Remove duplicates, sort, take the second from the end.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Track the largest and second largest in one pass.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Remove duplicates, sort, take the second from the end.

```python
def second_largest(nums):
    unique = sorted(set(nums))
    if len(unique) < 2:
        return None
    return unique[-2]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Track the largest and second largest in one pass.

```python
def second_largest(nums):
    first = second = None
    for x in nums:
        if first is None or x > first:
            first, second = x, first
        elif x != first and (second is None or x > second):
            second = x
    return second
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Remove duplicates, sort, take the second from the end. That costs O(n log n) time and O(n) space."

"I can do better: Track the largest and second largest in one pass. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/second-largest)

<a id="q16"></a>
### Q16. Rotate Right by k

🟡 Interview standard · *Slicing*

Return a new list rotated right by k: [1,2,3,4,5], k=2 → [4,5,1,2,3].

**Examples**

- `items = [1, 2, 3, 4, 5], k = 2` → `[4, 5, 1, 2, 3]`
- `items = [1, 2, 3], k = 4` → `[3, 1, 2]`
- `items = [], k = 3` → `[]`

In [ ]:
def rotate(items, k):
    """
    Return a new list rotated right by k: [1,2,3,4,5], k=2 → [4,5,1,2,3].
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rotate, [
    (([1, 2, 3, 4, 5], 2), [4, 5, 1, 2, 3]),
    (([1, 2, 3], 4), [3, 1, 2]),
    (([], 3), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Move the last item to the front, k times.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Two slices: the last k items followed by the rest (k taken modulo the length).

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Move the last item to the front, k times.

```python
def rotate(items, k):
    result = items[:]
    for _ in range(k):
        if result:
            last = result.pop()
            result.insert(0, last)
    return result
```

**Cost:** O(n · k) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Two slices: the last k items followed by the rest (k taken modulo the length).

```python
def rotate(items, k):
    if not items:
        return []
    k %= len(items)
    return items[-k:] + items[:-k] if k else items[:]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Move the last item to the front, k times. That costs O(n · k) time and O(n) space."

"I can do better: Two slices: the last k items followed by the rest (k taken modulo the length). That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/rotate-right-by-k)

<a id="q17"></a>
### Q17. Merge Two Sorted Lists

🟡 Interview standard · *Two indices*

Merge two sorted lists into one sorted list.

**Examples**

- `a = [1, 4, 7], b = [2, 3, 8]` → `[1, 2, 3, 4, 7, 8]`
- `a = [], b = [1]` → `[1]`
- `a = [1, 1], b = [1]` → `[1, 1, 1]`

In [ ]:
def merge_sorted(a, b):
    """
    Merge two sorted lists into one sorted list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(merge_sorted, [
    (([1, 4, 7], [2, 3, 8]), [1, 2, 3, 4, 7, 8]),
    (([], [1]), [1]),
    (([1, 1], [1]), [1, 1, 1]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Join them and sort the result.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Walk both lists with two indices, always taking the smaller head.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Join them and sort the result.

```python
def merge_sorted(a, b):
    merged = a + b
    merged.sort()
    return merged
```

**Cost:** O((m+n) log(m+n)) time, O(m+n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Walk both lists with two indices, always taking the smaller head.

```python
def merge_sorted(a, b):
    i = j = 0
    out = []
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            out.append(a[i])
            i += 1
        else:
            out.append(b[j])
            j += 1
    return out + a[i:] + b[j:]
```

**Cost:** O(m + n) time, O(m+n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Join them and sort the result. That costs O((m+n) log(m+n)) time and O(m+n) space."

"I can do better: Walk both lists with two indices, always taking the smaller head. That's O(m + n) time and O(m+n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/merge-two-sorted-lists)

<a id="q18"></a>
### Q18. Common Elements

🟡 Interview standard · *Nested loop vs set*

Return the values that appear in both lists, without duplicates, in the order they appear in `a`.

**Examples**

- `a = [1, 2, 3, 2, 4], b = [2, 4, 6]` → `[2, 4]`
- `a = [1, 2], b = [3]` → `[]`

In [ ]:
def common(a, b):
    """
    Return the values that appear in both lists, without duplicates, in the
    order they appear in a.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(common, [
    (([1, 2, 3, 2, 4], [2, 4, 6]), [2, 4]),
    (([1, 2], [3]), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each item of a, scan b; also check the result to avoid duplicates.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Put b in a set for O(1) lookups and track what you've already added.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** For each item of a, scan b; also check the result to avoid duplicates.

```python
def common(a, b):
    result = []
    for x in a:
        for y in b:
            if x == y and x not in result:
                result.append(x)
    return result
```

**Cost:** O(m · n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Put b in a set for O(1) lookups and track what you've already added.

```python
def common(a, b):
    in_b, seen, result = set(b), set(), []
    for x in a:
        if x in in_b and x not in seen:
            seen.add(x)
            result.append(x)
    return result
```

**Cost:** O(m + n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each item of a, scan b; also check the result to avoid duplicates. That costs O(m · n) time and O(k) space."

"I can do better: Put b in a set for O(1) lookups and track what you've already added. That's O(m + n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/common-elements)

<a id="q19"></a>
### Q19. Move Zeros to the End

🟡 Interview standard · *In-place write index*

Move every 0 to the end of the list in place, keeping the order of the other numbers. Return the list.

**Examples**

- `nums = [0, 1, 0, 3, 12]` → `[1, 3, 12, 0, 0]`
- `nums = [0]` → `[0]`
- `nums = [1, 2]` → `[1, 2]`

In [ ]:
def move_zeros(nums):
    """
    Move every 0 to the end of the list in place, keeping the order of the other
    numbers. Return the list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(move_zeros, [
    (([0, 1, 0, 3, 12],), [1, 3, 12, 0, 0]),
    (([0],), [0]),
    (([1, 2],), [1, 2]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count the zeros, remove them one by one with `remove`, then append that many zeros.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A write index copies every non-zero forward, then fill the rest with zeros.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Count the zeros, remove them one by one with `remove`, then append that many zeros.

```python
def move_zeros(nums):
    zeros = nums.count(0)
    for _ in range(zeros):
        nums.remove(0)
    for _ in range(zeros):
        nums.append(0)
    return nums
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A write index copies every non-zero forward, then fill the rest with zeros.

```python
def move_zeros(nums):
    w = 0
    for x in nums:
        if x != 0:
            nums[w] = x
            w += 1
    for i in range(w, len(nums)):
        nums[i] = 0
    return nums
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count the zeros, remove them one by one with `remove`, then append that many zeros. That costs O(n²) time and O(1) space."

"I can do better: A write index copies every non-zero forward, then fill the rest with zeros. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/move-zeros-to-the-end)

<a id="q20"></a>
### Q20. Transpose a Matrix

🟡 Interview standard · *Lists of lists · zip*

Swap rows and columns: [[1, 2, 3], [4, 5, 6]] → [[1, 4], [2, 5], [3, 6]].

**Examples**

- `matrix = [[1, 2, 3], [4, 5, 6]]` → `[[1, 4], [2, 5], [3, 6]]`
- `matrix = [[1]]` → `[[1]]`

In [ ]:
def transpose(matrix):
    """
    Swap rows and columns: [[1, 2, 3], [4, 5, 6]] → [[1, 4], [2, 5], [3, 6]].
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(transpose, [
    (([[1, 2, 3], [4, 5, 6]],), [[1, 4], [2, 5], [3, 6]]),
    (([[1]],), [[1]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build each new row by walking down a column with nested loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`zip(*matrix)` pairs up the columns.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Build each new row by walking down a column with nested loops.

```python
def transpose(matrix):
    rows, cols = len(matrix), len(matrix[0])
    result = []
    for c in range(cols):
        new_row = []
        for r in range(rows):
            new_row.append(matrix[r][c])
        result.append(new_row)
    return result
```

**Cost:** O(m · n) time, O(m · n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `zip(*matrix)` pairs up the columns.

```python
def transpose(matrix):
    return [list(col) for col in zip(*matrix)]
```

**Cost:** O(m · n) time, O(m · n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build each new row by walking down a column with nested loops. That costs O(m · n) time and O(m · n) space."

"I can do better: `zip(*matrix)` pairs up the columns. That's O(m · n) time and O(m · n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/lists/transpose-a-matrix)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What is the difference between append() and extend()?</b></summary>

`append(x)` adds x as one item (even if x is a list). `extend(iterable)` adds each item of the iterable. `[1].append([2, 3])` → `[1, [2, 3]]`; `[1].extend([2, 3])` → `[1, 2, 3]`.

</details>

<details>
<summary><b>2. sort() vs sorted()?</b></summary>

`list.sort()` sorts the list in place and returns None. `sorted()` takes any iterable and returns a new sorted list, leaving the original unchanged.

</details>

<details>
<summary><b>3. remove() vs pop() vs del?</b></summary>

`remove(x)` deletes the first item equal to x. `pop(i)` deletes the item at index i (default last) and returns it. `del a[i]` deletes by index or slice and returns nothing.

</details>

<details>
<summary><b>4. Shallow copy vs deep copy?</b></summary>

A shallow copy creates a new outer list but its items are the same objects as the original's. A deep copy recursively copies the nested objects too, so changes to inner lists don't affect the original.

</details>

<details>
<summary><b>5. Why is [[0] \* 3] \* 3 a bug?</b></summary>

The outer `* 3` repeats a reference to the same inner list three times. Changing `grid[0][0]` changes every row. Use `[[0] * 3 for _ in range(3)]`.

</details>

<details>
<summary><b>6. List vs tuple?</b></summary>

Lists are mutable and use square brackets; tuples are immutable and use parentheses. Tuples can be dict keys and signal 'this record shouldn't change'; lists are for collections that grow or change.

</details>

<details>
<summary><b>7. What is the time complexity of x in list?</b></summary>

O(n): Python compares x with each item in turn. Use a set or dict for O(1) membership tests.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Creating a list and reading items
- [ ] I can explain: Changing a list
- [ ] I can explain: Two names, one list (aliasing)
- [ ] I can explain: Looping over lists and building new ones
- [ ] I can explain: Sorting
- [ ] I can explain: Lists of lists (2-D lists)
- [ ] I can explain: Performance of list operations

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Rotate Right by k
- [ ] [Q17](#q17) Merge Two Sorted Lists
- [ ] [Q18](#q18) Common Elements
- [ ] [Q19](#q19) Move Zeros to the End
- [ ] [Q20](#q20) Transpose a Matrix

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.